In [1]:
%pip install -q -U "vllm>=0.17.0"
%pip show vllm

Name: vllm
Version: 0.32.0
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiohttp, anthropic, apache-tvm-ffi, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, einops, fastapi, fastsafetensors, filelock, flashinfer-python, huggingface_hub, humming-kernels, ijson, instanttensor, jsonschema, lark, llguidance, lm-format-enforcer, mcp, mistral_common, model-hosting-container-standards, msgspec, ninja, numba, numpy, nvidia-cudnn-frontend, nvidia-cutlass-dsl, nvtx, openai, opencv-python-headless, opentelemetry-api, opentelemetry-exporter-otlp, opentelemetry-sdk, opentelemetry-semantic-conventions-ai, oss-harmony, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, PyNvVideoCodec, python-json-lo

In [2]:
!nvidia-smi

Wed Oct 21 15:59:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   39C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os

os.environ["HF_TOKEN"] = input()

In [4]:
import hashlib
import json
import re

import pandas as pd
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams

In [5]:
MODEL_ID = "Qwen/Qwen3.6-35B-A3B-FP8"
DATASET_ID = "pameydorke/arcanumsearch-cross-platform"

NUM_QUERIES = 5
BATCH_SIZE = 64
MAX_RETRIES = 2

MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.90
MAX_NUM_SEQS = 128
MAX_NUM_BATCHED_TOKENS = 8192

BANNED_WORDS = ["arcanum", "experience", "reward"]
BANNED_RE = re.compile(r"\b(" + "|".join(BANNED_WORDS) + r")\b", re.IGNORECASE)

In [6]:
ds = load_dataset(DATASET_ID, split="train")

records = []
for i, row in enumerate(ds):
    link = (row.get("LINK") or "").strip()
    title = (row.get("TITLE") or "").strip()
    text = (row.get("TEXT") or "").strip()

    doc_id = hashlib.sha1(text[:200].encode("utf-8")).hexdigest()[:16]
    records.append(
        {
            "doc_id": doc_id,
            "link": link,
            "title": title,
            "text": text,
        }
    )

df_docs = pd.DataFrame(records).drop_duplicates("doc_id").reset_index(drop=True)
print("Documents:", len(df_docs))
print(df_docs.head(3).to_string())

README.md:   0%|          | 0.00/340 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  926kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1153 [00:00<?, ? examples/s]

Documents: 1153
             doc_id                                                          link                     title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [7]:
# PROMPT_TEMPLATE_V1 = """You are generating search queries for a retrieval tool used by an AI agent. The agent will type a query when it needs the information in the document below.

# Document title: {title}
# Document text:
# {text}

# Write {num_queries} concise, accurate search queries that should retrieve this exact document.

# Requirements:
# 1. Each query must be a natural-language search string of 1 to 14 words.
# 2. Include specific names, places, items, skills, quests, or concepts from the document so the query is unambiguous.
# 3. Cover different aspects of the document; do not paraphrase the same idea {num_queries} times.
# 4. Use normal capitalization and punctuation. Do NOT use quotation marks, boolean operators (AND/OR), site: filters, or markdown.
# 5. Do NOT mention "document", "article", "page", "text", or "wiki".
# 6. Never use the words "arcanum", "experience", or "reward" (case-insensitive).
# 7. Do not make every query just the document title.
# 8. Check spelling and use the exact names from the document.

# The queries should look like something a competent human or agent would actually type, not like a summarization task.

# Return ONLY a JSON array of exactly {num_queries} strings, for example:
# ["query one", "query two", "query three", "query four", "query five"]

# Queries:"""

PROMPT_TEMPLATE_V2 = """You are simulating the search queries that an AI agent sends to a retrieval tool. The agent reads a user's request (often informal, sometimes vague, sometimes misspelled), fixes obvious typos in ordinary English, and passes names it does not recognise straight through. The agent has NOT seen the document below and does not know what it contains.

Document title: {title}
Document text:
{text}

Write {num_queries} search queries that a capable agent could plausibly produce while trying to find this document.

What the agent knows: whatever a user would say about this topic, plus general knowledge.
What the agent does NOT know: coordinates, exact prices or stat values, the document's own wording, or which specific fact is on this page. Never put such information in a query.

Aim for this mix across the {num_queries} queries (for 5 queries: 2 / 2 / 1):
- about 40%: specific and correctly spelled, as a well-informed agent would write them;
- about 40%: specific but with ONE realistic degradation of a proper noun — a dropped apostrophe or hyphen, merged or split words, a dropped accent, a different plural, or a single plausible phonetic misspelling (e.g. "T'sen-Ang" -> "tsen gang", "Kerghan" -> "kergan");
- about 20%: a little vague — the topic or goal rather than the exact entity ("dwarven clan in the north", "who sold the stolen painting").

Rules:
1. Each query is 1 to 14 words, written like a search box entry, not a sentence. No question marks.
2. Each query must still point at this document, not at some generic topic page.
3. Cover different aspects of the document; do not restate one query five ways.
4. Degrade proper nouns only. Never misspell an ordinary English word — the agent fixes those. At most one degraded word per query.
5. No quotation marks, boolean operators, site: filters, or markdown.
6. Do NOT mention "document", "article", "page", "text", "wiki", "passage", "excerpt".
7. Never use the words "arcanum", "experience", "reward" (case-insensitive).
8. Do not copy the document title verbatim into every query.
9. A set where every query is perfectly spelled, or where every query is degraded, is wrong.

Good:
["Kerghan and Persephone painting stolen from Garringsburg", "Garringsburg gallery theft", "Kerghan and Persefone theft", "who bought the Kergan painting", "Garringsburg art missing"]
Bad (information unavailable before retrieval):
["Kerghan painting sold for 300 gold in 1884", "Pizarro painting at coordinates 45.2, 12.8"]

Return ONLY a JSON array of exactly {num_queries} strings, for example:
["query one", "query two", "query three", "query four", "query five"]

Queries:"""


def render_prompt(rec: dict, strict: bool = False) -> str:
    user = PROMPT_TEMPLATE_V2.format(
        title=rec["title"],
        text=rec["text"],
        num_queries=NUM_QUERIES,
    )
    if strict:
        user = (
            "IMPORTANT: Your previous response was rejected. "
            "Follow every rule exactly and return only the JSON array.\n\n" + user
        )
    return user

In [8]:
from vllm.sampling_params import StructuredOutputsParams

QUERY_SCHEMA = {
    "type": "array",
    "minItems": NUM_QUERIES,
    "maxItems": NUM_QUERIES,
    "items": {
        "type": "string",
        "minLength": 3,
        "maxLength": 200,
    },
}


def make_sampling_params(temperature: float = 0.6) -> SamplingParams:
    structured = StructuredOutputsParams(json=QUERY_SCHEMA)

    kwargs = dict(
        temperature=temperature,
        top_p=0.9,
        max_tokens=220,
        repetition_penalty=1.05,
        bad_words=[
            "arcanum", "Arcanum", "ARCANUM",
            "experience", "Experience",
            "reward", "Reward",
        ],
    )
    kwargs["structured_outputs"] = structured
    return SamplingParams(**kwargs)


sp_main = make_sampling_params(temperature=0.6)
sp_retry = make_sampling_params(temperature=0.3)

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    enable_prefix_caching=True,
    disable_log_stats=True,
    # Text-only generation from a multimodal checkpoint
    limit_mm_per_prompt={"image": 0, "video": 0},
)

tokenizer = llm.get_tokenizer()

(MainProcess pid=1717) INFO 10-21 16:00:10 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'max_num_batched_tokens': 8192, 'max_num_seqs': 128, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3.6-35B-A3B-FP8'}


config.json:   0%|          | 0.00/37.0k [00:00<?, ?B/s]

(MainProcess pid=1717) INFO 10-21 16:00:30 [model.py:733] Resolved architecture: Qwen3_5MoeForConditionalGeneration
(MainProcess pid=1717) INFO 10-21 16:00:30 [model.py:2152] Using max model len 4096


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

(MainProcess pid=1717) INFO 10-21 16:00:34 [model.py:951] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(MainProcess pid=1717) INFO 10-21 16:00:34 [model.py:1008] Disabled mm_prefix attention mode because multimodal inputs are configuration-disabled. Attention backends without mm_prefix support may now be selected.
(MainProcess pid=1717) INFO 10-21 16:00:34 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=1717) INFO 10-21 16:00:34 [config.py:688] Mamba cache mode is set to 'align' for Qwen3_5MoeForConditionalGeneration by default when prefix caching is enabled
(MainProcess pid=1717) INFO 10-21 16:00:34 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=1717) INFO 10-21 16:00:34 [compilation.py:317] Enabled custom fusions: norm_quant, act_q

In [9]:
def apply_chat_template(user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def parse_json_array(text: str):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
    return None


def clean_query(q: str) -> str:
    q = q.strip().strip('"').strip("'").strip()
    q = re.sub(r"\s+", " ", q)
    return q


def validate_queries(qs) -> tuple[bool, list[str] | str]:
    if qs == None:
        return False, "parse_error"

    if not isinstance(qs, list):
        return False, "not_list"

    if len(qs) != NUM_QUERIES:
        return False, f"not_{NUM_QUERIES}_queries"
    
    cleaned = []
    seen = set()

    for q in qs:
        if not isinstance(q, str):
            return False, "non_string"

        q = clean_query(q)
        words = q.split()

        if not (1 <= len(words) <= 14):
            return False, "bad_length"

        if BANNED_RE.search(q):
            return False, "banned_word"

        key = q.lower()
        if key in seen:
            return False, "duplicate"

        seen.add(key)
        cleaned.append(q)

    return True, cleaned


def generate_batch(batch, sp_main, sp_retry, max_retries=MAX_RETRIES):
    """
    Generate queries for a batch of records.
    Returns (successful_results: dict[doc_id -> list[str]], failed_records: list).
    """
    results = {}
    pending = list(batch)

    for attempt in range(max_retries + 1):
        if not pending:
            break

        sp = sp_main if attempt == 0 else sp_retry
        prompts = [apply_chat_template(render_prompt(r, strict=(attempt > 0))) for r in pending]

        outputs = llm.generate(prompts, sampling_params=sp)

        next_pending = []
        for rec, out in zip(pending, outputs):
            text = out.outputs[0].text
            qs = parse_json_array(text)

            ok, val = validate_queries(qs)

            if ok:
                results[rec["doc_id"]] = val
            else:
                next_pending.append(rec)

        pending = next_pending

    return results, pending

In [10]:
all_records = df_docs.to_dict("records")

results_by_id: dict[str, list[str]] = {}
failed_records: list[dict] = []

for start in tqdm(range(0, len(all_records), BATCH_SIZE), desc="Generating queries"):
    batch = all_records[start : start + BATCH_SIZE]

    results, failed_batch = generate_batch(batch, sp_main, sp_retry)

    results_by_id.update(results)
    failed_records.extend(failed_batch)

    if failed_batch:
        print(f"Batch {start}: {len(failed_batch)} document(s) failed validation")

print(f"Done. Successful: {len(results_by_id)} | Failed: {len(failed_records)}")

Generating queries:   0%|          | 0/19 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 64/64 [00:08<00:00,  7.17it/s, est. speed input: 7068.46 toks/s, output: 328.27 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.09it/s, est. speed input: 1787.69 toks/s, output: 111.73 toks/s]


...

Processed prompts: 100%|██████████| 2/2 [00:00<00:00,  3.38it/s, est. speed input: 2624.56 toks/s, output: 150.99 toks/s]

Batch 1088: 1 document(s) failed validation


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.06it/s, est. speed input: 1697.58 toks/s, output: 116.07 toks/s]

Done. Successful: 1149 | Failed: 4


In [11]:
failed_records

[{'doc_id': '4fbdb154b7f9fd52',
  'link': 'https://reddit.com/r/arcanum/comments/1jrzlca/high_res_patch/',
  'title': 'High Res patch',
  'text': "Original Post: High Res patch. Hello everyone :) \n\nI recently found Arcanum on my PC and immediately felt like playing it again.\n\n\n\nI also found a high-res patch on the internet, but it won't work. As soon as I change the size in the config, the game won't start anymore. However, after uninstalling it runs again.\n\nWhat am I doing wrong?\n\n\n\nMany thanks in advance! User 1: Are you using the UAP? It includes a high Res patch, and fixes a lot else too. It's very tried and tested, and if you have issues with it people here are experienced with it so can usually help trouble shoot. If you are using the UAP already, and adding a high Res patch on top that causes issues. It's exactly what I did when I started and that's when I was advised about the UAP having it already OP: https://preview.redd.it/olo1a84ukzse1.png?width=496&amp;format=p

In [12]:
successful = [r for r in all_records if r["doc_id"] in results_by_id]

df = pd.DataFrame(successful)
df["queries"] = df["doc_id"].map(results_by_id)

df = df[df["queries"].apply(lambda q: isinstance(q, list) and len(q) == NUM_QUERIES)].reset_index(drop=True)
print("Documents with 5 queries:", len(df))

pairs_df = (
    df[["doc_id", "link", "title", "text", "queries"]]
    .explode("queries")
    .rename(columns={"queries": "query"})
    .reset_index(drop=True)
)
pairs_ds = Dataset.from_pandas(pairs_df, preserve_index=False)

print(pairs_ds)

Documents with 5 queries: 1149
Dataset({
    features: ['doc_id', 'link', 'title', 'text', 'query'],
    num_rows: 5745
})


In [13]:
print("Query-count distribution:")
print(df["queries"].apply(len).value_counts().to_string())

all_queries = pairs_df["query"]
print("\nBanned-word matches:", int(all_queries.str.contains(BANNED_RE, regex=True).sum()))
print("Duplicate query strings:", int(all_queries.duplicated().sum()))
print("Duplicate query rate:", round(float(all_queries.duplicated().mean()), 4))

lengths = all_queries.str.split().str.len()
print("\nWord-count stats:")
print(lengths.describe().to_string())

print("\nRandom samples:")
for _, row in pairs_df.sample(min(15, len(pairs_df)), random_state=42).iterrows():
    print(f"- TITLE: {row['title']}")
    print(f"  QUERY: {row['query']}")

Query-count distribution:
queries
5    1149

Banned-word matches: 0
Duplicate query strings: 25
Duplicate query rate: 0.0044

Word-count stats:
count    5745.000000
mean        5.058312
std         1.071070
min         2.000000
25%         4.000000
50%         5.000000
75%         6.000000
max        11.000000

Random samples:
- TITLE: Z'an Al'urin
  QUERY: Z'an Al'urin dark elf follower recruitment requirements
- TITLE: Deliver Goods to Maximillian
  QUERY: Maximillian rightful heir Derholm throne
- TITLE: Gambling
  QUERY: gambing expert trainer gurin rockharrow
- TITLE: The Bog
  QUERY: bog location wheel clan
- TITLE: Dr. Edmund Craig
  QUERY: Dr Edmund Craig backstab master training
- TITLE: DX
  QUERY: makaal blessing vs velorians
- TITLE: Kree
  QUERY: Kree treasure container altar
- TITLE: Free J.T. Morgan
  QUERY: Free J.T. Morgan quest guide
- TITLE: Ancient Gods: Mazzerin's Mystery
  QUERY: Professor Aldous Buxington University of Tarant
- TITLE: Issues Installing the UAP
  

/tmp/ipykernel_1717/2608533778.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("\nBanned-word matches:", int(all_queries.str.contains(BANNED_RE, regex=True).sum()))


In [14]:
import numpy as np
from datasets import Dataset, DatasetDict

SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1

# ---- 1. Split unique doc_ids ----
unique_docs = pairs_df["doc_id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_docs)

n = len(unique_docs)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

train_ids = set(unique_docs[:n_train])
val_ids   = set(unique_docs[n_train : n_train + n_val])
test_ids  = set(unique_docs[n_train + n_val :])

# ---- 2. Build train first, collect its query set ----
train_df = pairs_df[pairs_df["doc_id"].isin(train_ids)].reset_index(drop=True)
train_q  = set(train_df["query"].str.lower())

# ---- 3. Val: drop pairs whose query already appeared in train ----
val_df = pairs_df[pairs_df["doc_id"].isin(val_ids)].copy()
val_df = val_df[~val_df["query"].str.lower().isin(train_q)].reset_index(drop=True)
val_q  = set(val_df["query"].str.lower())

# ---- 4. Test: drop pairs seen in train OR val ----
test_df = pairs_df[pairs_df["doc_id"].isin(test_ids)].copy()
test_df = test_df[~test_df["query"].str.lower().isin(train_q | val_q)].reset_index(drop=True)
test_q  = set(test_df["query"].str.lower())

# ---- 5. Leakage assertions (both doc and query level) ----
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert not (train_q & val_q),  f"query overlap train∩val: {len(train_q & val_q)}"
assert not (train_q & test_q), f"query overlap train∩test: {len(train_q & test_q)}"
assert not (val_q & test_q),   f"query overlap val∩test: {len(val_q & test_q)}"

# ---- 6. Report what was dropped ----
val_before = len(pairs_df[pairs_df["doc_id"].isin(val_ids)])
test_before = len(pairs_df[pairs_df["doc_id"].isin(test_ids)])

print(f"Dropped overlapping pairs — val: {val_before - len(val_df)} | test: {test_before - len(test_df)}")
print(f"Val docs with ≥1 query: {val_df['doc_id'].nunique()}/{len(val_ids)}")
print(f"Test docs with ≥1 query: {test_df['doc_id'].nunique()}/{len(test_ids)}")
print(f"Docs — train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")
print(f"Pairs — train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

# ---- 7. Wrap in HF Datasets ----
pairs_splits = DatasetDict({
    "train":      Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df,   preserve_index=False),
    "test":       Dataset.from_pandas(test_df,  preserve_index=False),
})
print(pairs_splits)

Dropped overlapping pairs — val: 2 | test: 7
Val docs with ≥1 query: 114/114
Test docs with ≥1 query: 116/116
Docs — train: 919 | val: 114 | test: 116
Pairs — train: 4595 | val: 568 | test: 573
DatasetDict({
    train: Dataset({
        features: ['doc_id', 'link', 'title', 'text', 'query'],
        num_rows: 4595
    })
    validation: Dataset({
        features: ['doc_id', 'link', 'title', 'text', 'query'],
        num_rows: 568
    })
    test: Dataset({
        features: ['doc_id', 'link', 'title', 'text', 'query'],
        num_rows: 573
    })
})


In [15]:
shuffled_ds = pairs_splits.shuffle(seed=42)

shuffled_ds["train"][0]

{'doc_id': 'caf20a68dc583cb1',
 'link': 'https://arcanum.fandom.com/wiki/Armour',
 'title': 'Armour',
 'text': 'Armour. Armour. File:Placeholder. Boots. Gloves. Helmets. Shields. Torso. Miscellaneous. Necklaces Rings',
 'query': 'armor equipment list'}

In [16]:
shuffled_ds.push_to_hub("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmppvx6h_z_.parquet    : 100%|##########|  120kB /  120kB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8/commit/a572f8d12ed452e4ed9e0908bc75cdc01f28b78f', commit_message='Upload dataset', commit_description='', oid='a572f8d12ed452e4ed9e0908bc75cdc01f28b78f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8'), pr_revision=None, pr_num=None)